# Running a simulation of a polymer on water

You will use [GROMACS](https://www.gromacs.org/) for simulate a simple system of PolyEthylene Glycol (PEG) in water. 

![peg.png](peg.png)

The minimum formula of it is H-(CH$_{2}$-O-CH$_2$)$_{n}$-H. Here, we would consider $n=12$.

In [ ]:
# Import basic packages
import nglview as ng
import pandas as pd
import mdtraj as md

In [ ]:
peg_file = 'peg.gro'
view = ng.show_file(peg_file,ext='gro')
display(view)

## Topology file
Gromacs uses a file called `.top` for reading the parameters of the system. The file has this structure:

```bash

[ defaults ]
; nbfunc    comb-rule    gen-pairs    fudgeLJ    fudgeQQ
    1         1            no           1.0     1.0

; Include forcefield parameters
#include "ff/charmm35r.itp"
#include "ff/peg.itp"
#include "ff/tip3p.itp"

[ system ]
; Name
    PEG

[ molecules ]
; Compound        #mols
    PEG             1

```

The file calls the force field parameters for the CHARMM force field (`charmm35r.itp`[version 35r](https://www.sciencedirect.com/science/article/pii/S0006349508701255)), the parameters of the peg molecule (`peg.itp`) and the water model TIP3p (`tip3.itp`[Jorgensen, _et al._](https://doi.org/10.1063/1.445869)).  


![tip3_water](tip3_water.png)

## Energy minimization parameters

First, we minimize the structure of the PEG molecule in vaccum. For this, we use the following file (Also save in `files/em.mdp`): 

```` bash
integrator = steep
emtol = 10
emstep = 0.0001
nsteps = 5000

nstenergy = 1000
nstxout = 100

cutoff-scheme = Verlet
coulombtype = PME
rcoulomb = 1
rvdw = 1
pbc = xyz

define = -DFLEXIBLE
````

Here, we ask GROMACS to use the steep-descendent algorithm for minmization (`integrator = steep`). The algortihm stops when it reaches the tolerance for the forces (`emtol = 10`) or the number of steps (`nsteps = 5000`). The initial step size for the minimization is controlled by `emstep = 0.0001`. Other parameters, `nstenergy = 1000` and `nstxout = 100` controlls how often energy and coordinates are saved. The parameters `cutoff-scheme`and `coulombtype` control how the electrostatic and non-bonded interactions are managed. For the moment, you do not need to worry about those as later in the course would be explained (😉). The cutoff for coulumb and van der Walls interactions are set to 1 nm and the periodic boundary conditions are on all directions. 

With the previous files, we could run our minimization!


By default, GROMACS first need to check all the data and aggrupated on an executable file `.tpr` which is used to run the simulation.

In [ ]:
!gmx grompp -f inputs/em.mdp -c peg.gro -p topol.top -o em-peg

After this step you must received a message that starts with: `:-) GROMACS - gmx grompp, 2025.4-conda_forge (-:`.

Now, we can run the minimization.

In [ ]:
! gmx mdrun -deffnm em-peg -v -nt 8

You should obtain as output something like:

```
Steepest Descents converged to Fmax < 10 in 2238 steps
Potential Energy  =  2.8141992e+02
Maximum force     =  9.9605055e+00 on atom 83
Norm of force     =  3.9791677e+00
```

We visualize the trajectory of the minimization.

In [ ]:
import mdtraj as md

traj = md.load("em-peg.trr", top="peg.gro")
view = ng.show_mdtraj(traj)
view

## Solvating the PEG

As a next step, we would solvate our molecule. First, we would put PEG at the center of a simulation cubic box of size 2.6 nm. Later, we would put our molecule on pre-equilibrated water box.

In [ ]:
!echo 0 0| gmx trjconv -f em-peg.gro -s em-peg.tpr -o peg-recentered.gro -center -pbc mol -box 2.6 2.6 2.6
!gmx solvate -cp peg-recentered.gro -cs spc216.gro -o peg-solvated.gro -p topol.top

You should have obtained something like:

```
Processing topology
Adding line for 547 solvent molecules with resname (SOL) to topology file (topol.top)
```

Now, we need to minimize the energy of the solvated structure. We would reuse the `.mdp` from the previous step.

In [ ]:
!gmx grompp -f inputs/em.mdp -c peg-solvated.gro -p topol.top -o em_solv
!gmx mdrun -deffnm em_solv -v -nt 8

We know visualize the minimized structure.

In [ ]:
file_solv = 'em_solv.gro'
view = ng.show_file(file_solv,ext='gro')
view.clear_representations()

# Add solute (PEG)
view.add_representation("licorice", selection="not water")

# Add solvent
view.add_representation("licorice", selection="water")
view

## Equilibration

The equilibration of the system would be done in two steps. First, in the $NVT$ ensemble and later on $NPT$ one. 

First for the $NVT$ simulation, we use the following `.mdp` parameter file.

``` bash
integrator = md
dt = 0.002
nsteps = 10000

nstenergy = 500
nstlog = 500
nstxout-compressed = 500

constraint-algorithm = lincs
constraints = hbonds
continuation = no

coulombtype = pme
rcoulomb = 1.0
rlist = 1.0

vdwtype = Cut-off
rvdw = 1.0

tcoupl = v-rescale
tau_t = 0.1 0.1
ref_t = 300 300
tc_grps = PEG Water

gen-vel = yes
gen-temp = 300
gen-seed = 65823

comm-mode = linear
comm-grps = PEG

```

Here, we use the leap-frog algorithm integrator (`integrator = md`). The temperature is controlled at 300 K by the v-rescale thermostate (For the moment, do not worry about it 😉). The initial velocities are also generated at 300 K. The lincs algorithm is used for keep the hydrogen bonds constant (This algorithm would be discussed later on the course). Finally, the `comm-mode` and `comm-grps` commands ensure that the PEG molecule remains centered in the box by removing its center-of-mass motion separately from the solvent. The file containing this parameters is `nvt-peg-h2o.mdp`

In [ ]:
!gmx grompp -f inputs/nvt-peg-h2o.mdp -c em_solv.gro -p topol.top -o nvt -maxwarn 1
!gmx mdrun -deffnm nvt -v -nt 8

Let's continue with the $NPT$ part of the equilibration. The `.mdp` file is almost the same but you would remove the part for the initial velocities as you continue with the simulation of the previous step. Also, you add the following parameters for the barostat.

``` bash
pcoupl = c-rescale
pcoupltype = isotropic
tau-p = 0.5
ref-p = 1.0
compressibility = 4.5e-5
```

We are using the barostat `c-rescale` (Do not worry about the details for now 😉). The only important details for you here are `ref-p` sets the reference pressure at 1 atm and `compressibility` specifies the isothermal compressibility of water, which is required for volume fluctuations to properly reflect the solvent properties. The file with this parameters is named: `npt-peg-h2o.mdp`. 

Run the $NPT$ part.

In [ ]:
!gmx grompp -f inputs/npt-peg-h2o.mdp -c nvt.gro -p topol.top -o npt -maxwarn 1
!gmx mdrun -deffnm npt -v -nt 8

If you reach this point without problems, you are ready to observe if the energy is indeed equilibrated after your simulations. By default, GROMACS save the information about it on a binary file with extension `.edr`. So, you need to use the function `gmx energy` to extract them and save it on `.xvg` files. From it, you would extract the potential energy only, although multiple quantities can be extracted; see [gmx energy](https://manual.gromacs.org/documentation/2023/onlinehelp/gmx-energy.html). 

The energy would be saved on a `.xvg` file. This format was designed to work with the xmgrace code, a legacy code from the 90's for plotting on linux. But do not worry, they recently release a new code called [plotXVG](https://github.com/AlexandriaChemistry/plotXVG) which allow us to plot the quantities with matplotlib. 

In [ ]:
!echo 7 | gmx energy -f em_solv.edr -o energy-em.xvg
!echo 7 | gmx energy -f nvt.edr -o energy-nvt.xvg
!echo 7 | gmx energy -f npt.edr -o energy-npt.xvg

In [ ]:
import plotxvg

plotxvg.plot(["energy-em.xvg", "energy-nvt.xvg","energy-npt.xvg"],
            ls=["solid", "solid","solid"], title=['Minimization', 'Equilibration NVT','Equilibration NPT'],
            panels='side',nofilelegends=True)

**Note**: The axis of the minimization step refeers to the steps not the time of simulation. 

## Production Run

The system is already equilibrated, so we can proceed to run the 'production' step of our simulation. This means that we would run a larger simulation for 200 ps in the $NVT$ ensemble. From that simulation, we can extract properties of interest. Either with the gromacs tools or with other like mdtraj or mdanalysis. 

In [ ]:
!gmx grompp -f inputs/production-peg-h2o.mdp -c npt.gro -p topol.top -o production -maxwarn 1
!gmx mdrun -deffnm production -v -nt 8

One important aspect is to remove PBC to avoid problems at the time of analysis. To do that, you can use a GROMACS tool called `gmx trjconv`. This avoid break bonds or too streched ones.

In [ ]:
!echo 0 0| gmx trjconv -s production.tpr -f production.xtc -o prod_center.xtc -center -pbc mol 

Let's visualize your trajectory!

In [ ]:
traj = md.load("prod_center.xtc", top="nvt.gro")
view = ng.show_mdtraj(traj)
view.clear_representations()

# Add solute (PEG)
view.add_representation("ball+stick", selection="PEG")

# Add solvent
view.add_representation("licorice", selection="water")
view

So, congratulations! 🎉🎉🎉🎉 You made your first molecular simulation on GROMACS. Now, with the results and trajectory files. Solve the tasks in the exercise sheet. 